# NLP & Computer Vision Mastery

## Chapter 17: Fine-tuning AlexNet in Code

Read this chapter: [https://books.vizuara.ai/read/nlp-cv-mastery/finetuning-alexnet-in-code](https://books.vizuara.ai/read/nlp-cv-mastery/finetuning-alexnet-in-code)

---

**Generated file.** Extracted automatically from the book's own text. Do not hand-edit; edit the book instead.

Code cells appear in the order the blocks appear in the chapter, and may assume names defined in earlier cells or earlier chapters. Nothing here has been executed.

In [ ]:

# --- context (added to make this file runnable) ---
# Block 3 iterates a DataLoader the chapter assumes you have already built.
from context import train_loader

In [ ]:
# --- block 1 ---
import torch.nn as nn
from torchvision import models

# 1. Load AlexNet with its pretrained ImageNet weights
alexnet = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# 2. FREEZE everything — no gradients, no updates
for param in alexnet.parameters():
    param.requires_grad = False

# 3. Replace the head: the last layer is classifier[6], a Linear(4096 -> 1000)
alexnet.classifier[6] = nn.Linear(4096, 5)   # 5 flower classes, requires_grad=True by default

In [ ]:
# --- block 2 ---
import torch

criterion = nn.CrossEntropyLoss()   # softmax + categorical cross-entropy, fused

# Give the optimizer ONLY the parameters that still require grad — the new head
trainable = [p for p in alexnet.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-3)

In [ ]:
# --- block 3 ---
for epoch in range(30):
    alexnet.train()
    for images, labels in train_loader:      # batches of (224×224×3 images, class ids 0–4)
        optimizer.zero_grad()                # clear last batch's gradients
        logits = alexnet(images)             # forward: (batch, 5) raw scores
        loss = criterion(logits, labels)     # softmax + cross-entropy
        loss.backward()                      # backward: gradients ONLY reach the head
        optimizer.step()                     # update the 20,485 head weights